# Text-to-Outfit — Notebook 1: emparejamiento outfit↔texto + extracción de embeddings

**Notebook autocontenido**: no hace falta pegar nada de `Recommendation_Engine_v2_accelerated.ipynb`.
Las primeras celdas (0–7) replican exactamente el setup mínimo de ese notebook (carga de
embeddings de texto, metadatos de ítem, definición de la arquitectura, instanciación del
modelo) para que esto se pueda ejecutar de arriba a abajo en un kernel nuevo.

**Requisitos previos en disco** (ya deberías tenerlos, son los que generó tu pipeline original):
- `glove_text_embeddings.npy` + `glove_text_embeddings_index.json`
- `glove_text_features.csv`
- Directorio de imágenes (`IMAGES_DIR`)
- `best_ce.pt`
- `train.json`, `valid.json`, `test.json` (partición `nondisjoint`)
- `polyvore_outfit_titles.json`

**Ajusta `POLYVORE_PATH` en la primera celda de código a tu ruta local.**


## 0. Imports y configuración

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import os, json, random
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"  # AJUSTA A TU RUTA LOCAL
IMAGES_DIR    = POLYVORE_PATH + "images/"
OUTPUT_DIR    = POLYVORE_PATH + "data/processed/"

TEXT_EMBED_DIM = 100   # dim de los vectores GloVe (media de tokens, SIN PCA)
IMG_SIZE       = 224
MAX_LEN        = 8     # longitud máxima del outfit -- debe coincidir con el checkpoint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 1. Embeddings de texto precalculados por ítem (media de GloVe, sin PCA)

In [49]:
TEXT_EMBEDDINGS_PATH = OUTPUT_DIR + "glove_text_embeddings.npy"
TEXT_INDEX_PATH      = OUTPUT_DIR + "glove_text_embeddings_index.json"

text_matrix = np.load(TEXT_EMBEDDINGS_PATH)          # (N, 100) -- sin PCA
with open(TEXT_INDEX_PATH) as f:
    text_index = json.load(f)                        # {pos: item_id}

assert text_matrix.shape[1] == TEXT_EMBED_DIM, (
    f"TEXT_EMBED_DIM ({TEXT_EMBED_DIM}) no coincide con la matriz cargada ({text_matrix.shape[1]})."
)

item_id2textvec = {v: text_matrix[int(k)].astype(np.float32) for k, v in text_index.items()}
print(f"Embeddings de texto cargados: {len(item_id2textvec)} ítems, dim={text_matrix.shape[1]}")

Embeddings de texto cargados: 251008 ítems, dim=100


## 2. Metadatos por ítem: imagen, categoría

In [50]:
df = pd.read_csv(OUTPUT_DIR + "glove_text_features.csv")
df['item_id'] = df['item_id'].astype(str)

cat_encoder = LabelEncoder()
df['cat_id'] = cat_encoder.fit_transform(df['sub_category'])
NUM_CATEGORIES = df['cat_id'].nunique()
print(f"Categorías: {NUM_CATEGORIES}")  # debe salir 118 para que encaje con best_ce.pt

item_id2img_path = {iid: os.path.join(IMAGES_DIR, f"{iid}.jpg") for iid in df['item_id']}
item_id2cat      = dict(zip(df['item_id'], df['cat_id']))

missing_text = [iid for iid in df['item_id'] if iid not in item_id2textvec]
print(f"Ítems sin vector de texto: {len(missing_text)}")

Categorías: 118
Ítems sin vector de texto: 0


## 3. Arquitectura — Stage 1: encoder multimodal end-to-end

In [51]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out = out + residual
        return self.relu(out)


def make_layer(in_planes, planes, num_blocks, stride=1):
    downsample = None
    if stride != 1 or in_planes != planes * BasicBlock.expansion:
        downsample = nn.Sequential(
            nn.Conv2d(in_planes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(planes * BasicBlock.expansion),
        )
    layers = [BasicBlock(in_planes, planes, stride=stride, downsample=downsample)]
    in_planes = planes * BasicBlock.expansion
    for _ in range(1, num_blocks):
        layers.append(BasicBlock(in_planes, planes))
    return nn.Sequential(*layers), in_planes


class SmallResNet18(nn.Module):
    def __init__(self, embedding_size=64):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        in_planes = 64
        self.layer1, in_planes = make_layer(in_planes, 64,  num_blocks=2, stride=1)
        self.layer2, in_planes = make_layer(in_planes, 128, num_blocks=2, stride=2)
        self.layer3, in_planes = make_layer(in_planes, 256, num_blocks=2, stride=2)

        self.avgpool  = nn.AvgPool2d(kernel_size=7)
        self.flatten  = nn.Flatten()
        self.fc_embed = nn.Linear(256 * 2 * 2, embedding_size)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        return self.fc_embed(x)


class TextEncoder(nn.Module):
    def __init__(self, embedding_size=64, input_dim=TEXT_EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(input_dim, embedding_size)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, text_vec):
        return self.relu(self.proj(text_vec))


class CategoricalEncoder(nn.Module):
    def __init__(self, num_categories, embedding_size=64):
        super().__init__()
        self.embedding = nn.Embedding(num_categories, embedding_size)

    def forward(self, cat_ids):
        return self.embedding(cat_ids)


class ItemEncoder(nn.Module):
    def __init__(self, num_categories, common_dim=128):
        super().__init__()
        self.image_encoder = SmallResNet18(embedding_size=common_dim)
        self.text_encoder  = TextEncoder(embedding_size=common_dim)
        self.cat_encoder   = CategoricalEncoder(num_categories, embedding_size=common_dim)
        self.norm = nn.LayerNorm(common_dim, eps=1e-6)

    def forward(self, images, text_vecs, cat_ids):
        img_emb = self.image_encoder(images)
        txt_emb = self.text_encoder(text_vecs)
        cat_emb = self.cat_encoder(cat_ids)
        item_emb = img_emb + txt_emb + cat_emb
        return self.norm(item_emb)

## 4. Arquitectura — TAE (Type-Aware Gate)

In [52]:
class TypeAwareGate(nn.Module):
    def __init__(self, num_categories, d_model):
        super().__init__()
        self.type_embedding = nn.Embedding(num_categories, d_model)
        self.gamma_proj = nn.Linear(d_model, d_model)
        self.beta_proj  = nn.Linear(d_model, d_model)

    def forward(self, x, cat_ids):
        t = self.type_embedding(cat_ids)
        gamma = torch.sigmoid(self.gamma_proj(t))
        beta  = self.beta_proj(t)
        return x * gamma + beta

## 5. Arquitectura — Stage 2: Transformer + modelo completo

In [53]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                                dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        ffn_out = self.ffn(x)
        return self.norm2(x + self.drop2(ffn_out))


class TAETransformerOutfitModel(nn.Module):
    def __init__(self, num_categories, d_model=128, n_heads=4, ffn_dim=256,
                 n_layers=2, dropout=0.1, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.item_encoder = ItemEncoder(num_categories, common_dim=d_model)
        self.type_gate     = TypeAwareGate(num_categories, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)

        self.ce_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid(),
        )
        self.fitb_head = nn.Sequential(
            nn.Linear(d_model * 2, 128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def encode_items(self, images, text_vecs, cat_ids):
        B, L = images.shape[0], images.shape[1]
        images_flat = images.reshape(B * L, 3, IMG_SIZE, IMG_SIZE)
        text_flat   = text_vecs.reshape(B * L, TEXT_EMBED_DIM)
        cats_flat   = cat_ids.reshape(B * L)

        item_emb = self.item_encoder(images_flat, text_flat, cats_flat)
        item_emb = self.type_gate(item_emb, cats_flat)
        item_emb = item_emb.reshape(B, L, -1)
        return item_emb

    def encode_outfit(self, images, text_vecs, cat_ids, padding_mask=None):
        x = self.encode_items(images, text_vecs, cat_ids)
        x = self.dropout(x)

        batch_size = x.shape[0]
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)

        if padding_mask is not None:
            cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=x.device)
            key_padding_mask = torch.cat([cls_mask, padding_mask], dim=1)
        else:
            key_padding_mask = None

        for layer in self.encoder_layers:
            x = layer(x, key_padding_mask=key_padding_mask)

        cls_out = F.normalize(x[:, 0, :], p=2, dim=-1)
        return cls_out

    def call_ce(self, images, text_vecs, cat_ids, padding_mask=None):
        cls = self.encode_outfit(images, text_vecs, cat_ids, padding_mask)
        return self.ce_head(cls)

## 6. Pipeline de datos — carga de imágenes y construcción de tensores de outfit

In [54]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def load_image(path):
    img = Image.open(path).convert("RGB")
    return image_transform(img)


def get_item_raw(item_id):
    """Devuelve (path_imagen, vector_texto, cat_id) crudos para un item_id."""
    item_id = str(item_id)
    return item_id2img_path[item_id], item_id2textvec[item_id], item_id2cat[item_id]


def build_outfit_tensors(item_ids, max_len=MAX_LEN):
    """
    Devuelve (images, text_vecs, cat_ids, mask) con padding a max_len.
    - images    : (max_len, 3, 224, 224)
    - text_vecs : (max_len, TEXT_EMBED_DIM)
    - cat_ids   : (max_len,)
    - mask      : (max_len,) bool, True donde hay padding
    """
    n = min(len(item_ids), max_len)
    images, texts, cats = [], [], []
    for iid in item_ids[:n]:
        path, text_vec, cat_id = get_item_raw(iid)
        images.append(load_image(path))
        texts.append(torch.from_numpy(text_vec))
        cats.append(cat_id)

    pad_needed = max_len - n
    images += [torch.zeros(3, IMG_SIZE, IMG_SIZE)] * pad_needed
    texts  += [torch.zeros(TEXT_EMBED_DIM)] * pad_needed
    cats   += [0] * pad_needed
    mask    = [False] * n + [True] * pad_needed

    images_t = torch.stack(images)
    texts_t  = torch.stack(texts).float()
    cats_t   = torch.tensor(cats, dtype=torch.long)
    mask_t   = torch.tensor(mask, dtype=torch.bool)
    return images_t, texts_t, cats_t, mask_t

## 7. Instanciar el modelo (mismos hiperparámetros que en el entrenamiento original)

In [55]:
D_MODEL  = 128
N_HEADS  = 4
FFN_DIM  = 256
N_LAYERS = 2
DROPOUT  = 0.1

model = TAETransformerOutfitModel(
    num_categories=NUM_CATEGORIES,
    d_model=D_MODEL, n_heads=N_HEADS, ffn_dim=FFN_DIM, n_layers=N_LAYERS,
    dropout=DROPOUT, max_len=MAX_LEN,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {n_params:,}")

Parámetros totales: 3,305,026


---
## A partir de aquí: las celdas del bloque Text-to-Outfit propiamente dicho

### A. Cargar el checkpoint entrenado (congelado, solo inferencia)

In [56]:
CKPT_PATH_CE = POLYVORE_PATH + "checkpoints_RE/best_ce.pt"  # ajusta si hace falta

ckpt = torch.load(CKPT_PATH_CE, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
for p in model.parameters():
    p.requires_grad = False  # congelado: esto es la Opción 1, no fine-tuning del RE

print(f"Checkpoint cargado (época {ckpt['epoch']}, métrica {ckpt['best_metric']:.4f})")

C:\Users\angel\AppData\Local\Temp\ipykernel_3896\600843551.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH_CE, map_location=device)


Checkpoint cargado (época 36, métrica 0.8942)


### B. Reconstruir `outfit_id -> [item_ids ordenados]` y su split

In [57]:
def build_outfit_item_map(json_paths_with_split):
    outfit_items = {}
    outfit_split = {}
    for path, split in json_paths_with_split:
        with open(path) as f:
            outfits = json.load(f)
        for outfit in outfits:
            set_id = str(outfit["set_id"])
            items_sorted = sorted(outfit["items"], key=lambda it: it["index"])
            outfit_items[set_id] = [str(it["item_id"]) for it in items_sorted]
            outfit_split[set_id] = split
    return outfit_items, outfit_split


OUTFIT_SPLIT_PATHS = [
    (POLYVORE_PATH + "nondisjoint/train.json", "train"),
    (POLYVORE_PATH + "nondisjoint/valid.json", "valid"),
    (POLYVORE_PATH + "nondisjoint/test.json",  "test"),
]
outfit_items, outfit_split = build_outfit_item_map(OUTFIT_SPLIT_PATHS)
print(f"Outfits indexados desde train/valid/test.json: {len(outfit_items)}")

Outfits indexados desde train/valid/test.json: 68306


### C. Emparejar cada outfit con su texto (`title`, fallback a `url_name`)

In [58]:
OUTFIT_TITLES_PATH = POLYVORE_PATH + "polyvore_outfit_titles.json"  # ajusta si hace falta

with open(OUTFIT_TITLES_PATH) as f:
    outfit_titles_raw = json.load(f)


def clean_text(s):
    return " ".join(s.strip().split())


def get_outfit_text(entry):
    title = clean_text(entry.get("title", ""))
    if title:
        return title
    return clean_text(entry.get("url_name", ""))


t2o_samples = []
skipped_no_text = 0
skipped_no_items = 0
skipped_missing_item_data = 0

for set_id, entry in outfit_titles_raw.items():
    text = get_outfit_text(entry)
    if not text:
        skipped_no_text += 1
        continue

    item_ids = outfit_items.get(set_id)
    if not item_ids:
        skipped_no_items += 1
        continue

    if not all(iid in item_id2img_path and iid in item_id2textvec and iid in item_id2cat
               for iid in item_ids):
        skipped_missing_item_data += 1
        continue

    t2o_samples.append({
        "set_id": set_id,
        "item_ids": item_ids,
        "text": text,
        "split": outfit_split.get(set_id, "unknown"),
    })

print(f"Muestras Text-to-Outfit utilizables: {len(t2o_samples)}")
print(f"  Descartadas por falta de texto (title y url_name vacíos): {skipped_no_text}")
print(f"  Descartadas por no encontrar el outfit en train/valid/test.json: {skipped_no_items}")
print(f"  Descartadas por falta de datos de algún ítem (imagen/texto/categoría): {skipped_missing_item_data}")

t2o_train = [s for s in t2o_samples if s["split"] == "train"]
t2o_valid = [s for s in t2o_samples if s["split"] == "valid"]
t2o_test  = [s for s in t2o_samples if s["split"] == "test"]
print(f"train={len(t2o_train)}  valid={len(t2o_valid)}  test={len(t2o_test)}")

Muestras Text-to-Outfit utilizables: 68306
  Descartadas por falta de texto (title y url_name vacíos): 0
  Descartadas por no encontrar el outfit en train/valid/test.json: 0
  Descartadas por falta de datos de algún ítem (imagen/texto/categoría): 0
train=53306  valid=5000  test=10000


### D. Extraer embeddings de outfit congelados (una sola vez, se cachean)

**Nota de rendimiento:** un forward por outfit (batch=1). Con ~57k outfits es lento uno-a-uno
por el overhead de lanzar kernels pequeños repetidamente. Se deja simple aquí primero para
validar que el pairing es correcto antes de optimizar.

In [59]:
@torch.no_grad()
def compute_outfit_embedding(item_ids):
    images, texts, cats, mask = build_outfit_tensors(item_ids, max_len=MAX_LEN)
    images = images.unsqueeze(0).to(device)
    texts  = texts.unsqueeze(0).to(device)
    cats   = cats.unsqueeze(0).to(device)
    mask   = mask.unsqueeze(0).to(device)
    emb = model.encode_outfit(images, texts, cats, mask)  # (1, d_model), ya L2-normalizado
    return emb.squeeze(0).cpu()


model.eval()
for s in tqdm(t2o_samples, desc="Extrayendo embeddings de outfit (congelado)"):
    s["outfit_emb"] = compute_outfit_embedding(s["item_ids"])

T2O_CACHE_PATH = POLYVORE_PATH + "checkpoints_RE/t2o_samples_with_embeddings.pt"
torch.save(t2o_samples, T2O_CACHE_PATH)
print(f"Guardado: {T2O_CACHE_PATH}")

Extrayendo embeddings de outfit (congelado): 100%|██████████| 68306/68306 [25:13<00:00, 45.12it/s] 


Guardado: ./polyvore_kaggle/polyvore_outfits/checkpoints_RE/t2o_samples_with_embeddings.pt


---
### Siguiente paso

Continúa con `text_to_outfit_02_query_encoder_training.ipynb`, que carga
`t2o_samples_with_embeddings.pt` y entrena el encoder de queries GloVe + MLP con pérdida
contrastiva InfoNCE. Ese notebook ya NO depende de este (no vuelve a tocar imágenes ni el
Recommendation Engine), así que puedes cerrar este kernel después de guardar la caché.